# Mamba-3 논문 읽기, 세 가지 변경을 노트북에서 재현

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/mamba3_ssm_mechanisms.ipynb)

[Mamba-3 논문](https://arxiv.org/abs/2603.15569)
(arXiv 2603.15569)의 세 메커니즘을 작은 독립 실험으로 확인합니다.

- 이산화: LTI·LTV 기준해와 비교해 최대 오차와 수렴 차수를 봅니다.
- 패리티: 세 전이 변형의 홀짝 정확도와 길이 일반화를 비교합니다.
- 디코딩 비용: 길이별 캐시·상태와 SISO·MIMO 산술 강도를 잽니다.

**학습된 Mamba-3를 재현하지 않고 메커니즘만 봅니다.**
장치에 따라 시간 수치는 달라집니다. 해석은 출력된 값을 보세요.

`런타임 → 런타임 유형 변경 → T4 GPU`를 권장합니다.
CPU도 동작합니다. 비용 측정은 CUDA → MPS → CPU 순으로 고르고,
패리티 학습은 CPU·float32로 고정합니다.
기본 런타임의 torch·numpy만 사용하며 추가 설치는 없습니다.

In [ ]:
import math
import statistics
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("torch  :", torch.__version__)
print("numpy  :", np.__version__)
print("device :", device)
if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(device)
elif device.type == "mps":
    gpu_name = "Apple GPU (MPS)"
else:
    gpu_name = "none (CPU)"
print("gpu    :", gpu_name)
print("parity : CPU / float32")

## 1. 이산화 — LTI 표와 LTV 표

지수-오일러와 지수-사다리꼴을 같은 입력·초기값으로 비교합니다.
LTI는 해석해, LTV는 원래 시변 ODE의 float64 RK4를 기준으로 씁니다.
`delta`를 줄일 때 최대 절대 오차와 `order`가 어떻게 바뀌는지 보세요.
각 표의 λ=1 검사는 두 갱신식이 같은지도 확인합니다.

In [ ]:
def a(t):
    return -1.0 - 0.5 * np.sin(t)


def forcing(t):
    """b(t) x(t), with b=1 and x(t)=cos(3t)."""
    return np.cos(3.0 * t)


def reference(end=10.0, dt=1e-5):
    """RK4 on the time-varying ODE; no frozen a(t)."""
    count = round(end / dt)
    times = np.arange(count + 1, dtype=np.float64) * dt
    left_a, left_u = a(times), forcing(times)
    midpoints = times[:-1] + dt / 2
    middle_a, middle_u = a(midpoints), forcing(midpoints)
    h = np.zeros(count + 1, dtype=np.float64)
    for i in range(count):
        k1 = left_a[i] * h[i] + left_u[i]
        k2 = middle_a[i] * (h[i] + dt * k1 / 2) + middle_u[i]
        k3 = middle_a[i] * (h[i] + dt * k2 / 2) + middle_u[i]
        k4 = left_a[i + 1] * (h[i] + dt * k3) + left_u[i + 1]
        h[i + 1] = h[i] + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6
    return h


def discretize(delta, a, forcing, lam=None, end=10.0):
    """lam=None: exp-Euler; otherwise exp-trapezoidal."""
    times = (
        np.arange(round(end / delta) + 1, dtype=np.float64)
        * delta
    )
    h = np.zeros_like(times)
    decay, u = np.exp(delta * a(times)), forcing(times)
    for t in range(1, len(times)):
        if lam is None:
            h[t] = decay[t] * h[t - 1] + delta * u[t]
        else:
            h[t] = (decay[t] * h[t - 1]
                    + (1 - lam) * delta * decay[t] * u[t - 1]
                    + lam * delta * u[t])
    return h


def print_table(title, a, exact):
    print(title)
    print(" delta   Euler_max_abs  order   "
          "Trapezoid_max_abs  order")
    deltas = (0.2, 0.1, 0.05, 0.025, 0.0125)
    previous = None
    for delta in deltas:
        euler = discretize(delta, a, forcing)
        trapezoid = discretize(delta, a, forcing, lam=0.5)
        assert np.array_equal(
            euler, discretize(delta, a, forcing, lam=1.0)
        )
        times = np.arange(len(euler), dtype=np.float64) * delta
        truth = exact(times)
        errors = [np.max(np.abs(h - truth))
                  for h in (euler, trapezoid)]
        if previous is None:
            orders = ["   --", "   --"]
        else:
            rates = np.log2(np.array(previous) / errors)
            orders = [f"{rate:5.2f}" for rate in rates]
        print(f"{delta:7.4f} {errors[0]:14.7e} {orders[0]}"
              f" {errors[1]:19.7e} {orders[1]}")
        previous = errors
    print(f"Observed order ({deltas[-2]:g} -> "
          f"{deltas[-1]:g}): "
          f"Euler={rates[0]:.2f}, Trapezoid={rates[1]:.2f}")
    print("lambda=1 equals Euler: "
          "assert passed for every delta")

In [ ]:
discretization_start = time.perf_counter()
print_table(
    "LTI: h'=-h+cos(3*t), h(0)=0, t in [0,10]; "
    "reference: analytic solution",
    lambda t: -np.ones_like(t),
    lambda t: (
        np.cos(3 * t) + 3 * np.sin(3 * t) - np.exp(-t)
    ) / 10,
)

In [ ]:
dt = 1e-5
truth = reference(dt=dt)
print()
print_table(
    "LTV: h'=(-1-0.5*sin(t))*h+cos(3*t), h(0)=0, "
    "t in [0,10]; "
    "reference: float64 RK4 dt=1e-5 at shared grid points",
    a,
    lambda t: truth[np.rint(t / dt).astype(int)],
)
elapsed = time.perf_counter() - discretization_start
print(f"elapsed_s={elapsed:.2f}")

## 2. 패리티 — 세 변형, 시드 3개

실수 감쇠(`real`), 고정 회전(`rope`), 입력 의존 회전(`complex`)을
같은 16쌍 상태·투영·초기화·학습 데이터로 비교합니다.
CPU·float32에서 각각 1,600스텝 학습하므로 시간이 걸릴 수 있습니다.
길이 32·256의 모든 위치 정확도와 각 시드·중앙값을 출력에서 보세요.

In [ ]:
class DiagonalSSM(nn.Module):
    """Same 16 real pairs and projections in all variants."""

    def __init__(self, variant, size=16):
        super().__init__()
        self.variant = variant
        self.embed = nn.Embedding(2, size)
        self.delta = nn.Linear(size, size)
        self.input = nn.Linear(size, 2 * size)
        self.output = nn.Linear(2 * size, 1)
        self.A_log = nn.Parameter(torch.full((size,), -8.0))
        # Learned, token-independent initial state.
        self.initial = nn.Parameter(torch.cat((
            torch.ones(size), torch.zeros(size)
        )))
        self.register_buffer(
            "theta",
            torch.logspace(-1, math.log10(math.pi), size),
        )
        nn.init.normal_(self.delta.weight, std=0.5)

    def forward(self, bits):
        z = self.embed(bits)
        delta = F.softplus(self.delta(z))
        alpha = torch.exp(-delta * self.A_log.exp())
        if self.variant == "real":
            angle = torch.zeros_like(delta)
        elif self.variant == "rope":
            angle = self.theta.expand_as(delta)
        else:
            angle = delta * self.theta
        cosine, sine = alpha * angle.cos(), alpha * angle.sin()
        u_real, u_imag = self.input(z).chunk(2, dim=-1)
        real, imag = self.initial.chunk(2)
        real = real.expand(bits.shape[0], -1)
        imag = imag.expand_as(real)
        states = []
        # Real coordinate pairs implement complex arithmetic.
        for t in range(bits.shape[1]):
            real, imag = (
                cosine[:, t] * real - sine[:, t] * imag
                + u_real[:, t],
                sine[:, t] * real + cosine[:, t] * imag
                + u_imag[:, t],
            )
            states.append(torch.cat((real, imag), dim=-1))
        return self.output(
            torch.stack(states, dim=1)
        ).squeeze(-1)


def sample(batch, length, generator):
    bits = torch.randint(
        2, (batch, length), generator=generator, device="cpu"
    )
    return bits, (bits.cumsum(dim=1) % 2).float()


@torch.no_grad()
def accuracy(model, length, seed, batch=512):
    bits, labels = sample(
        batch, length,
        torch.Generator().manual_seed(10000 + seed),
    )
    return (
        ((model(bits) >= 0) == labels.bool())
        .float().mean().item() * 100
    )


def train(variant, seed, steps=1600):
    torch.manual_seed(seed)
    model = DiagonalSSM(variant).to("cpu", dtype=torch.float32)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    generator = torch.Generator().manual_seed(1000 + seed)
    start = time.perf_counter()
    for _ in range(steps):
        bits, labels = sample(64, 32, generator)
        optimizer.zero_grad(set_to_none=True)
        loss = F.binary_cross_entropy_with_logits(
            model(bits), labels
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    return model, time.perf_counter() - start

In [ ]:
parity_start = time.perf_counter()
torch.set_num_threads(1)
print(f"device=cpu torch={torch.__version__} dtype=float32",
      flush=True)
print("Adam lr=0.01 steps=1600 batch=64 "
      "train_length=32 state=16 pairs")
print("Accuracy: every position, "
      "512 fresh sequences/length, chance=50%")
print("variant seed length32(%) length256(%) train_s",
      flush=True)
for variant in ("real", "rope", "complex"):
    scores = []
    for seed in (0, 1, 2):
        model, elapsed = train(variant, seed)
        row = (accuracy(model, 32, seed),
               accuracy(model, 256, seed))
        scores.append(row)
        print(f"{variant:7s} {seed:4d} {row[0]:11.2f} "
              f"{row[1]:12.2f} {elapsed:7.2f}", flush=True)
    medians = [statistics.median(row[i] for row in scores)
               for i in (0, 1)]
    print(f"{variant:7s} med  {medians[0]:11.2f} "
          f"{medians[1]:12.2f}", flush=True)
print("All seeds reported, including failures; "
      "this is a toy optimization experiment.")
print(f"elapsed_s={time.perf_counter() - parity_start:.2f}")

## 3. 디코딩 비용 — 길이별 표와 SISO/MIMO 표

길이별 KV 캐시·순환 상태 바이트와 토큰당 시간을 비교합니다.
SISO·랭크 4 MIMO의 FLOP/byte는 이상적 융합의 이동량 하한입니다.
시간은 융합하지 않은 PyTorch 연산의 중앙값·최소·최대입니다.
투영·캐시 추가·모델 전체 비용은 제외하며 출력된 값을 보세요.

In [ ]:
HEADS, P, N = 16, 64, 128
DTYPE = torch.float16
REPEATS, TRIALS = 100, 5


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def measure(step, device):
    """Discard a 100-update warm-up, then measure five more."""
    samples = []
    for trial in range(TRIALS + 1):
        synchronize(device)
        start = time.perf_counter()
        for _ in range(REPEATS):
            step()
        synchronize(device)
        if trial:
            samples.append(
                (time.perf_counter() - start) * 1000 / REPEATS
            )
    return (statistics.median(samples),
            min(samples), max(samples))


def recurrent_step(batch, rank, device):
    """h: (B,H,N,P), B/C: (B,H,N,R), x: (B,H,P,R)."""
    def random(*shape):
        return (
            torch.randn(*shape, device=device, dtype=DTYPE)
            * 0.01
        )

    h = random(batch, HEADS, N, P)
    B = random(batch, HEADS, N, rank)
    C = random(batch, HEADS, N, rank)
    x = random(batch, HEADS, P, rank)
    alpha = torch.full(
        (batch, HEADS, 1, 1), 0.99, device=device, dtype=DTYPE
    )

    def step():
        # Same unfused operations for SISO (R=1) and MIMO (R=4).
        h.mul_(alpha)
        h.add_(B @ x.transpose(-1, -2))
        return C.transpose(-1, -2) @ h

    return step, h.numel() * h.element_size()


def operation_counts(batch, rank):
    """FMA=2 FLOPs; ideal fused traffic, not measured DRAM."""
    groups = batch * HEADS
    state = N * P
    # Scale h + rank-R update + readout; GEMMs: 2*m*n*k.
    flops = groups * (2 * state + 4 * state * rank)
    # Read/write h once; read alpha/B/C/x; write y (R,P).
    elements = groups * (
        2 * state + 1 + 2 * N * rank + 2 * P * rank
    )
    return (
        flops,
        elements * torch.empty((), dtype=DTYPE).element_size(),
    )


def timing_text(values):
    median, low, high = values
    return f"{median:10.4f} [{low:.4f}, {high:.4f}]"

In [ ]:
@torch.inference_mode()
def decode_lengths():
    torch.set_num_threads(1)
    torch.manual_seed(0)
    print(f"device={device} torch={torch.__version__} "
          f"dtype={DTYPE}", flush=True)
    print("d_model=1024 heads=16 P=64 N=128; "
          "warmup=1 trials=5 repeats=100")
    print("Batch=1: cache/state bytes; "
          "median [min, max] ms per token")
    print("     L   attention_KV_B  SISO_state_B   "
          "attention_ms [range]       SISO_ms [range]",
          flush=True)
    q = torch.randn(1, HEADS, 1, P, device=device, dtype=DTYPE)
    for length in (1024, 4096, 16384, 65536):
        k = torch.randn(
            1, HEADS, length, P, device=device, dtype=DTYPE
        )
        v = torch.randn_like(k)
        # Newest query: all cached keys are visible.
        attention = measure(
            lambda: F.scaled_dot_product_attention(q, k, v),
            device,
        )
        step, state_bytes = recurrent_step(1, 1, device)
        recurrent = measure(step, device)
        kv_bytes = (k.numel() + v.numel()) * k.element_size()
        print(f"{length:6d} {kv_bytes:16d} {state_bytes:13d} "
              f"{timing_text(attention)}  "
              f"{timing_text(recurrent)}", flush=True)
        del k, v, step


decode_start = time.perf_counter()
decode_lengths()

In [ ]:
@torch.inference_mode()
def decode_mimo():
    print("\nBatch=128: arithmetic intensity uses "
          "ideal fused bytes (read+write).")
    print("Unfused PyTorch timings include temporaries/extra "
          "traffic; bytes are a lower bound.")
    print("variant R        FLOPs  ideal_RW_bytes  FLOP/byte   "
          "median_ms [min, max]", flush=True)
    for name, rank in (("SISO", 1), ("MIMO", 4)):
        step, _ = recurrent_step(128, rank, device)
        flops, traffic = operation_counts(128, rank)
        timings = measure(step, device)
        print(f"{name:7s} {rank} {flops:12d} {traffic:15d} "
              f"{flops / traffic:10.3f} "
              f"{timing_text(timings)}", flush=True)
        del step
    print("No projections, cache append, or trained model; "
          "MIMO need not be faster here.")


decode_mimo()
print(f"elapsed_s={time.perf_counter() - decode_start:.2f}")